# Object Recognition with Open Images v7 and Webcam

In this notebook we prepare a dataset from **Open Images v7** using **FiftyOne**, crop the annotated objects with **OpenCV**, then adapt a **ResNet50** model with transfer learning and fine-tuning in **TensorFlow/Keras** to recognize everyday objects on a live webcam feed.

**Steps:**
1. Import libraries
2. Choose target classes
3. Download the train and validation subsets
4. Crop objects and check class balance *(next)*
5. Transfer learning with a frozen ResNet50 *(next)*
6. Fine-tuning the top layers *(next)*
7. Evaluate and run the live webcam demo *(next)*

In [1]:
import fiftyone as fo
import fiftyone.zoo as foz

In [2]:
print(fo.list_datasets())

You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information
['train_dataset', 'validation_dataset']


## 1. Target classes

We pick **30 everyday classes** that are likely to appear in front of a webcam:

- **Electronics:** mobile phone, laptop, keyboard, mouse, headphones, monitor, remote control, camera
- **Office items:** book, pen, scissors, calculator
- **Kitchen and tableware:** mug, bottle, coffee cup, plate, bowl, spoon, fork, knife
- **Personal items:** glasses, watch, backpack, clock
- **Fruit:** banana, apple, orange
- **People:** person, human face, human hand

Class names must match the Open Images names exactly (capitalized as in the official list). The next cell prints the number of classes as a sanity check.

In [3]:
labels = [
    "Mobile phone", "Laptop", "Computer keyboard", "Computer mouse",
    "Headphones", "Computer monitor", "Remote control", "Camera",
    "Book", "Pen", "Scissors", "Calculator",
    "Mug", "Bottle", "Coffee cup", "Plate", "Bowl", "Spoon", "Fork", "Knife",
    "Glasses", "Watch", "Backpack", "Clock",
    "Banana", "Apple", "Orange",
    "Person", "Human face", "Human hand",
]
print(len(labels))

30


## 2. Download the dataset

We download the **train** split of Open Images v7 with bounding-box labels (`detections`), **one class at a time**, so every class gets a similar number of images instead of letting frequent classes (Person, Clothing, ...) dominate.

- `classes=[label]`: only images containing the current class (one class per iteration)
- `max_samples=1000`: images per class (about 30,000 images in total for 96 classes)
- `shuffle=True`, `seed=i`: random but reproducible selection
- `dataset_name`: a temporary dataset per class, merged into `train_dataset` and then deleted

> The first run downloads large metadata files (about 4.8 GB), so it can take several minutes. Later runs use the local cache in `~/fiftyone/open-images-v7`. If the download is interrupted, run the cell again: already downloaded images are not downloaded twice.

In [4]:
if fo.dataset_exists("train_dataset"):
    fo.delete_dataset("train_dataset")
train_ds = fo.Dataset("train_dataset", persistent=True)

for i, label in enumerate(labels):
    tmp = f"tmp_train_{i}"
    if fo.dataset_exists(tmp):
        fo.delete_dataset(tmp)

    ds = foz.load_zoo_dataset(
        "open-images-v7",
        split="train",
        classes=[label],
        label_types=["detections"],
        max_samples=1000,
        shuffle=True,
        seed=i,
        dataset_name=tmp,
    )
    train_ds.add_samples(ds)
    ds.delete()
    print(i, label, len(train_ds))

 100% |█████████████████| 1000/1000 [4.3m elapsed, 0s remaining, 4.5 files/s]      
Dataset info written to '/home/amirm/fiftyone/open-images-v7/info.json'
Loading 'open-images-v7' split 'train'
 100% |███████████████| 1000/1000 [4.3s elapsed, 0s remaining, 247.0 samples/s]      
Dataset 'tmp_train_0' created
 100% |███████████████| 1000/1000 [4.1s elapsed, 0s remaining, 258.0 samples/s]      
0 Mobile phone 1000
Found 5 images, downloading the remaining 995
 100% |███████████████████| 995/995 [4.9m elapsed, 0s remaining, 3.7 files/s]      
Dataset info written to '/home/amirm/fiftyone/open-images-v7/info.json'
Loading 'open-images-v7' split 'train'
 100% |███████████████| 1000/1000 [7.9s elapsed, 0s remaining, 111.0 samples/s]      
Dataset 'tmp_train_1' created
 100% |███████████████| 1000/1000 [8.8s elapsed, 0s remaining, 106.2 samples/s]      
1 Laptop 2000
Found 80 images, downloading the remaining 920
 100% |███████████████████| 920/920 [3.8m elapsed, 0s remaining, 3.3 files/s]  

## 3. Download the validation split

We download the **validation** split of Open Images v7 the same way as the train split: one class at a time, same target classes, same label type (`detections`), so the model can be evaluated on images it has never seen during training.

- `split="validation"`: the validation part of Open Images v7
- `classes=[label]`: only images containing the current class (one class per iteration)
- `max_samples=300`: images per class (about 9,000 images in total, about 30% of the training size)
- `shuffle=True`, `seed=i`: random but reproducible selection
- `dataset_name`: a temporary dataset per class, merged into `validation_dataset` and then deleted

> The train and validation splits are separate in Open Images, so there is no overlap between them. The validation metadata is much smaller than the train metadata, so this step is faster than the previous download. Rare classes may have fewer than 300 images.

In [7]:
if fo.dataset_exists("validation_dataset"):
    fo.delete_dataset("validation_dataset")
val_ds = fo.Dataset("validation_dataset", persistent=True)

for i, label in enumerate(labels):
    tmp = f"tmp_val_{i}"
    if fo.dataset_exists(tmp):
        fo.delete_dataset(tmp)

    ds = foz.load_zoo_dataset(
        "open-images-v7",
        split="validation",
        classes=[label],
        label_types=["detections"],
        max_samples=300,
        shuffle=True,
        seed=i,
        dataset_name=tmp,
    )
    val_ds.add_samples(ds)
    ds.delete()
    print(i, label, len(val_ds))

Only found 105 (<300) samples matching your requirements
 100% |███████████████████| 105/105 [24.3s elapsed, 0s remaining, 6.1 files/s]      
Dataset info written to '/home/amirm/fiftyone/open-images-v7/info.json'
Loading 'open-images-v7' split 'validation'
 100% |█████████████████| 105/105 [704.8ms elapsed, 0s remaining, 149.0 samples/s]      
Dataset 'tmp_val_0' created
 100% |█████████████████| 105/105 [604.6ms elapsed, 0s remaining, 174.6 samples/s]      
0 Mobile phone 105
Only found 52 (<300) samples matching your requirements
 100% |█████████████████████| 52/52 [16.1s elapsed, 0s remaining, 3.6 files/s]      
Dataset info written to '/home/amirm/fiftyone/open-images-v7/info.json'
Loading 'open-images-v7' split 'validation'
 100% |███████████████████| 52/52 [546.5ms elapsed, 0s remaining, 95.2 samples/s]      
Dataset 'tmp_val_1' created
 100% |███████████████████| 52/52 [733.7ms elapsed, 0s remaining, 70.9 samples/s]      
1 Laptop 157
Only found 63 (<300) samples matching your 

## 4. Load the saved datasets

Both datasets were created with `persistent=True`, so they stay in the FiftyOne database after the download. In any later session we only need to load them, without downloading anything again.

- `fo.load_dataset("train_dataset")`: the train split (one class at a time, up to 1000 images per class)
- `fo.load_dataset("validation_dataset")`: the validation split (one class at a time, up to 300 images per class)
- `len(...)`: number of images, used as a quick sanity check

> Do not run the download cells (sections 2 and 3) again: they delete and rebuild the datasets. From now on, start from this cell.

> The counts are lower than the targets (30 classes x 1000 and x 300) because some classes have fewer images available than `max_samples`, especially in the much smaller validation split.

In [5]:
train_ds = fo.load_dataset('train_dataset')
val_ds = fo.load_dataset('validation_dataset')
print(len(train_ds), len(val_ds))

26142 2673


## 5. Inspect the dataset

We open the FiftyOne app to look at the images and their labels before preparing the training data.

- `fo.launch_app(train_ds)`: opens the app (in the notebook, or at `http://localhost:5151` in the browser)
- Each image is shown with its bounding boxes and class names (the `ground_truth` field)
- The sidebar on the left can filter the images by label, for example only `Laptop`

> This cell is only for exploring the data and is not needed to run the rest of the notebook. Close the app with `session.close()` when finished.

In [13]:
session = fo.launch_app(train_ds)

The `show_confidence` App config option has been removed and no longer has any effect. Label attribute visibility is controlled per attribute in the App's sidebar
The `show_index` App config option has been removed and no longer has any effect. Label attribute visibility is controlled per attribute in the App's sidebar
The `show_label` App config option has been removed and no longer has any effect. Label attribute visibility is controlled per attribute in the App's sidebar
The `show_confidence` App config option has been removed and no longer has any effect. Label attribute visibility is controlled per attribute in the App's sidebar
The `show_index` App config option has been removed and no longer has any effect. Label attribute visibility is controlled per attribute in the App's sidebar
The `show_label` App config option has been removed and no longer has any effect. Label attribute visibility is controlled per attribute in the App's sidebar


## 6. Keep only the target classes

The images carry the labels of every object that Open Images annotated, not only our 30 classes (for example `Table` or `Clothing`). We create two views that keep only the boxes of the 30 target classes.

- `F("label").is_in(labels)`: keeps a box only if its class is in our `labels` list
- `only_matches=True`: drops images that have no box left after filtering
- A view does not copy or change the dataset on disk, it only filters what we see

> Both counts are the same as the full datasets (26,142 train and 2,673 validation), because every image was downloaded for at least one target class. The views will be used in the next step to crop the objects.

In [14]:
from fiftyone import ViewField as F
train_view = train_ds.filter_labels("ground_truth", F("label").is_in(labels), only_matches= True)
val_view = val_ds.filter_labels("ground_truth", F("label").is_in(labels), only_matches= True)
print(len(train_view), len(val_view))

26142 2673


## 7. Look at one sample

Before building the labels, we look at what one sample contains.

- `train_view.select_fields("ground_truth")`: reads only the `ground_truth` field of each image (faster)
- `view.first()`: takes the first image of the view
- `sample.ground_truth.detections`: the list of boxes of this image, each with a `label` (class name) and a `bounding_box` (`[x, y, width, height]`, relative values between 0 and 1)

> One image can have several boxes, and several boxes can have the same class. We only need the class names here, the box coordinates are not used by the classifier.

In [39]:
view= train_view.select_fields("ground_truth")
sample = view.first()
print(sample.filepath)
print(sample.ground_truth.detections)
print([s.label for s in sample.ground_truth.detections])

/home/amirm/fiftyone/open-images-v7/train/data/63652712e3a82343.jpg
[<Detection: {
    'id': '6ae752608d14ff959aa0d5ff',
    'attributes': {},
    'tags': [],
    'label': 'Mobile phone',
    'bounding_box': [0.735625, 0.696429, 0.156875, 0.12781900000000002],
    'mask': None,
    'mask_path': None,
    'confidence': None,
    'index': None,
    'IsOccluded': True,
    'IsTruncated': False,
    'IsGroupOf': False,
    'IsDepiction': False,
    'IsInside': False,
}>]
['Mobile phone']


## 8. Build multi-hot labels

We do not crop the objects. Instead, the model looks at the **whole image** and predicts which of the 30 classes are present (multi-label classification).

- `labels_index`: maps every class name to its position in `labels` (for example `Mobile phone` is `0`). The order of `labels` must not change after training.
- `build_array`: for every image it creates a vector of 30 zeros and sets the position of each class that appears in the image to `1.0`
- The result is `train_y` with shape `(26142, 30)` and `val_y` with shape `(2673, 30)`: one row per image, one column per class

> This is **multi-hot** encoding: an image with a phone and a laptop has two ones in its vector. Because several classes can be true at the same time, the model will use a `sigmoid` output and `binary_crossentropy` loss instead of `softmax`.

In [36]:
labels_index = {name: i for i, name in enumerate(labels)}
print(labels_index)

{'Mobile phone': 0, 'Laptop': 1, 'Computer keyboard': 2, 'Computer mouse': 3, 'Headphones': 4, 'Computer monitor': 5, 'Remote control': 6, 'Camera': 7, 'Book': 8, 'Pen': 9, 'Scissors': 10, 'Calculator': 11, 'Mug': 12, 'Bottle': 13, 'Coffee cup': 14, 'Plate': 15, 'Bowl': 16, 'Spoon': 17, 'Fork': 18, 'Knife': 19, 'Glasses': 20, 'Watch': 21, 'Backpack': 22, 'Clock': 23, 'Banana': 24, 'Apple': 25, 'Orange': 26, 'Person': 27, 'Human face': 28, 'Human hand': 29}


In [45]:
import numpy as np
def build_array (view):
    paths, targets = [], []
    for sample in view.select_fields("ground_truth").iter_samples():
        multi_hot = np.zeros(len(labels), dtype = 'float32')
        for l in sample.ground_truth.detections:
            multi_hot[labels_index[l.label]] = 1.0
        paths.append(sample.filepath)
        targets.append(multi_hot)
    return paths, np.array(targets)
    
train_paths, train_y = build_array(train_view)
val_paths, val_y = build_array(val_view)
print(train_y.shape, val_y.shape)

(26142, 30) (2673, 30)


## 9. Merge duplicate images

Each class was downloaded separately, so an image that contains several target classes can appear more than once in `train_paths`, each time with only one class set to `1.0`. The model would see the same image with different, incomplete labels. Here we merge the rows of the same image into one.

- `merge_by_path`: goes through all (path, label vector) pairs and keeps one row per image path
- `np.maximum(merged[p], row)`: combines two label vectors of the same image, a class is `1.0` if it is `1.0` in any of them (logical OR)
- `len(train_paths)` vs `len(set(train_paths))`: the total number of rows against the number of unique images. If the two numbers differ, the dataset had duplicates

> After merging, `train_y` and `val_y` have one row per unique image, so their first dimension is smaller than before if duplicates were found. Some objects can still be missing from the labels: FiftyOne only loaded the boxes of the class requested for each download, so a class that was never requested for an image stays `0.0`.

In [47]:
def merge_by_path(paths, y):
    merged = {}
    for p, row in zip(paths, y):
        merged[p] = np.maximum(merged[p], row) if p in merged else row.copy()
    return list(merged.keys()), np.array(list(merged.values()))

print(len(train_paths), len(set(train_paths))) 
train_paths, train_y = merge_by_path(train_paths, train_y)
val_paths, val_y = merge_by_path(val_paths, val_y)
print(train_y.shape, val_y.shape)

22388 22388
(22388, 30) (2093, 30)


## Import TensorFlow and Keras

Import TensorFlow and Keras to build and configure the transfer learning model.

In [48]:
import tensorflow as tf
from tensorflow import keras
from keras import layers

## Load Pretrained ResNet50

Load the ResNet50 architecture with ImageNet pretrained weights.
The original classification head is removed using `include_top=False`,
and the input size is set to `224 × 224 × 3` for RGB images.

In [52]:
base_model = keras.applications.resnet50.ResNet50(weights= 'imagenet', 
                                                  include_top= False, 
                                                  input_shape= (224, 224, 3))

## Freeze the ResNet50 Backbone

Freeze all layers of the pretrained ResNet50 backbone to preserve
the learned ImageNet features during the initial transfer learning stage.

This allows us to train a new classification head on top of the
pretrained feature extractor without updating the backbone weights.

In [53]:
base_model.trainable = False
base_model.summary()

Model: "resnet50"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_pad           │ (None, 230, 230,  │          0 │ input_layer_1[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,472 │ conv1_pad[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pad           │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1_pool          │ (None, 56, 56,    │          0 │ pool1_pad[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      4,160 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        256 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,928 │ conv2_block1_1_r… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_bn   │ (None, 56, 56,    │        256 │ conv2_block1_2_c… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_relu │ (None, 56, 56,    │          0 │ conv2_block1_2_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_conv │ (None, 56, 56,    │     16,640 │ pool1_pool[0][0]  │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_conv │ (None, 56, 56,    │     16,640 │ conv2_block1_2_r… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_0_c… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_3_bn   │ (None, 56, 56,    │      1,024 │ conv2_block1_3_c

 Total params: 23,587,712 (89.98 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 23,587,712 (89.98 MB)

## Prepare the Data Pipeline

We use TensorFlow's `tf.data` API to efficiently load and preprocess images.

- `load_img`: reads, decodes and resizes images to `224 × 224`.
- `make_ds`: creates datasets from image paths and labels.
- `shuffle(10_000)`: randomly shuffles training data.
- `map()`: preprocesses images in parallel.
- `batch(32)`: groups images into batches of 32.
- `prefetch(AUTOTUNE)`: improves data loading performance.

In [55]:
autotune = tf.data.AUTOTUNE
batch = 32

def load_img (path, y):
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels= 3)
    img = tf.image.resize(img, (224, 224))
    return img, y

def make_ds (paths, y, training):
    ds = tf.data.Dataset.from_tensor_slices((paths, y))
    if training:
        ds = ds.shuffle(10_000)
    return ds.map(load_img, num_parallel_calls= autotune).batch(batch).prefetch(autotune)

## Create Datasets

Create separate datasets for training and validation:

- `train_tf`: shuffled training dataset.
- `val_tf`: validation dataset without shuffling.

Both datasets use the same preprocessing and batching pipeline.

In [56]:
train_tf = make_ds(train_paths, train_y, True)
val_tf = make_ds(val_paths, val_y, False)

## 10. Data augmentation

To reduce overfitting, the training images are randomly changed on the fly: the model sees a slightly different version of each image in every epoch.

- `RandomFlip("horizontal")`: mirrors the image left to right
- `RandomRotation(0.1)`: rotates by up to about 36 degrees (0.1 of a full turn)
- `RandomZoom(0.1)`: zooms in or out by up to 10%
- `RandomTranslation(0.1, 0.1)`: shifts the image by up to 10% in each direction
- `RandomContrast(0.1)`: changes the contrast slightly

> These layers are only active during training. At validation and prediction time they do nothing.

In [59]:
data_augmentation = keras.models.Sequential([
    layers.RandomFlip("Horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomContrast(0.1)
])

## 11. Build the model

We put a new classification head on top of the pretrained ResNet50 (transfer learning, phase 1).

- `Input` -> data augmentation -> `preprocess_input` (the scaling ResNet50 was trained with) -> ResNet50 backbone
- `GlobalAveragePooling2D`: turns the feature maps into one vector per image
- `Dense(512, activation="relu")`: a hidden layer that learns combinations of the features
- `Dropout(0.2)`: randomly switches off 20% of the units during training to reduce overfitting
- `Dense(30, activation="sigmoid")`: one independent probability per class

> The ResNet50 backbone is frozen (`base_model.trainable = False`), so only the new head is trained. The output uses `sigmoid` because several classes can be present in the same image.

In [69]:
inputs = keras.Input(shape= (224, 224, 3))
input_gen = data_augmentation(inputs)
x = keras.applications.resnet50.preprocess_input(inputs)
x = base_model(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation= 'relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(30, activation= 'sigmoid')(x)

model = keras.models.Model(inputs, outputs)

## 12. Compile and train the head

- `loss=binary_crossentropy`: the loss for multi-label classification (each class is a yes/no question)
- `Adam(learning_rate=1e-3)`: the learning rate for the new head
- `AUC(multi_label=True)`: how well the model ranks images that contain a class above images that do not, averaged over the classes (it does not depend on a threshold)
- `Precision` and `Recall`: computed with a threshold of 0.5 (a class counts as present when its probability is above 0.5)
- `epochs=5`: the number of passes over the training data

> Accuracy is not used here: with 30 classes and mostly zeros in each label vector, it would look high even for a poor model.

In [70]:
model.compile(loss= keras.losses.binary_crossentropy,
             optimizer= keras.optimizers.Adam(learning_rate= 1e-3),
             metrics=[keras.metrics.AUC(multi_label=True, name="auc"),
                      keras.metrics.Precision(name="precision"),
                      keras.metrics.Recall(name="recall")])

In [71]:
his = model.fit(train_tf, validation_data= val_tf, epochs= 5)

Epoch 1/5


I0000 00:00:1793633633.852613    2832 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_74993__.187


699/700 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - auc: 0.9405 - loss: 0.0948 - precision: 0.7082 - recall: 0.5104

I0000 00:00:1793633692.396352    2830 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_74993__.187


700/700 ━━━━━━━━━━━━━━━━━━━━ 93s 105ms/step - auc: 0.9405 - loss: 0.0947 - precision: 0.7084 - recall: 0.5106 - val_auc: 0.9698 - val_loss: 0.0796 - val_precision: 0.8066 - val_recall: 0.5071
Epoch 2/5
700/700 ━━━━━━━━━━━━━━━━━━━━ 53s 76ms/step - auc: 0.9675 - loss: 0.0724 - precision: 0.7888 - recall: 0.5979 - val_auc: 0.9670 - val_loss: 0.0847 - val_precision: 0.8398 - val_recall: 0.4995
Epoch 3/5
700/700 ━━━━━━━━━━━━━━━━━━━━ 82s 75ms/step - auc: 0.9721 - loss: 0.0669 - precision: 0.8034 - recall: 0.6251 - val_auc: 0.9698 - val_loss: 0.0771 - val_precision: 0.8038 - val_recall: 0.5559
Epoch 4/5
700/700 ━━━━━━━━━━━━━━━━━━━━ 53s 76ms/step - auc: 0.9766 - loss: 0.0618 - precision: 0.8142 - recall: 0.6581 - val_auc: 0.9661 - val_loss: 0.0807 - val_precision: 0.8112 - val_recall: 0.5387
Epoch 5/5
700/700 ━━━━━━━━━━━━━━━━━━━━ 53s 76ms/step - auc: 0.9794 - loss: 0.0583 - precision: 0.8212 - recall: 0.6777 - val_auc: 0.9619 - val_loss: 0.0790 - val_precision: 0.8032 - val_recall: 0.5782


## 13. Evaluate the head

We evaluate the model on the training set and on the validation set (images it has never seen).

| Metric | Train | Validation |
|--------|:-----:|:----------:|
| Loss | 0.05 | 0.08 |
| AUC | 0.987 | 0.962 |
| Precision | 0.865 | 0.803 |
| Recall | 0.692 | 0.578 |

- The AUC is high on both sets, so the model ranks the classes well.
- The gap between train and validation is small but visible (AUC 0.987 vs 0.962), a sign of mild overfitting.
- The recall is low because the threshold is 0.5: the model misses many classes that are present. Lowering the threshold (for example to 0.3) raises the recall and lowers the precision.
- In the training log the validation AUC stopped improving after about 3 epochs, so more epochs of the frozen head will not help much. The next step is fine-tuning.

In [79]:
train_result = model.evaluate(train_tf)
print(f"Train:\nLoss: {(train_result[0]):.2f}\nAccuracy: {(train_result[1] * 100):.2f}")

700/700 ━━━━━━━━━━━━━━━━━━━━ 50s 71ms/step - auc: 0.9875 - loss: 0.0490 - precision: 0.8696 - recall: 0.6975
Train:
Loss: 0.05
Accuracy: 98.75


In [78]:
val_result = model.evaluate(val_tf)
print(f"Validation:\nLoss: {(val_result[0]):.2f}\nAccuracy: {(val_result[1] * 100):.2f}")

66/66 ━━━━━━━━━━━━━━━━━━━━ 5s 68ms/step - auc: 0.9619 - loss: 0.0790 - precision: 0.8032 - recall: 0.5782
Validation:
Loss: 0.08
Accuracy: 96.19
